[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/upm-classes/ai_medical_imaging_2026/blob/main/notebooks/student/NB-X4_federated_fedavg.ipynb)

# NB-X4 · Challenge: federated learning from scratch (FedAvg, NumPy)
**Day 5 · optional · about 35 minutes · CPU only, no deep-learning libraries.** Three "hospitals" keep their data local; only model weights travel. We implement FedAvg on a logistic regression with synthetic data and see what happens when the hospitals' populations differ (**non-IID**).

In [ ]:
import numpy as np, matplotlib.pyplot as plt
SEED = 0; rng = np.random.default_rng(SEED)
def make_site(n, shift, pos_rate, d=5, seed=0):
    r = np.random.default_rng(seed); y = (r.random(n) < pos_rate).astype(float)
    X = r.normal(0, 1, (n, d)) + shift + np.outer(y, np.array([1.2, 0.8, 0, 0, 0])) ; return X, y
SITES = [make_site(400, np.array([0, 0, 0, 0, 0.]), 0.30, seed=1), make_site(400, np.array([0.5, 0, 0, 0, 0.]), 0.30, seed=2), make_site(400, np.array([-0.5, 0, 0, 0, 0.]), 0.30, seed=3)]
Xte, yte = make_site(2000, np.zeros(5), 0.30, seed=99)
sigmoid = lambda z: 1 / (1 + np.exp(-np.clip(z, -30, 30)))
def auc(y, s):
    order = np.argsort(s); ranks = np.empty(len(s)); ranks[order] = np.arange(1, len(s) + 1); n1 = y.sum(); n0 = len(y) - n1
    return (ranks[y == 1].sum() - n1 * (n1 + 1) / 2) / (n1 * n0)
print([ (len(s[1]), round(s[1].mean(), 2)) for s in SITES])

## 1 · Local training
**TODO 1:** `local_sgd(w, X, y, epochs, lr)` = full-batch gradient descent on the logistic loss starting from `w` (the last entry of `w` is the bias). Gradient: `Xb.T @ (sigmoid(Xb @ w) - y) / n`, with `Xb = [X, 1]`.

In [ ]:
def add_bias(X): return np.hstack([X, np.ones((len(X), 1))])
def local_sgd(w, X, y, epochs=5, lr=0.5):
    w = w.copy()
    # TODO 1: epochs of full-batch gradient descent on the logistic loss (see the formula above)
    return w

In [ ]:
w0 = np.zeros(6); w1 = local_sgd(w0, *SITES[0], epochs=50)
Xb = add_bias(SITES[0][0]); l0 = -np.mean(SITES[0][1] * np.log(sigmoid(Xb @ w0) + 1e-9) + (1 - SITES[0][1]) * np.log(1 - sigmoid(Xb @ w0) + 1e-9)); l1 = -np.mean(SITES[0][1] * np.log(sigmoid(Xb @ w1) + 1e-9) + (1 - SITES[0][1]) * np.log(1 - sigmoid(Xb @ w1) + 1e-9))
assert l1 < l0 - 0.05, "local training should reduce the loss"
print(f"loss {l0:.3f} -> {l1:.3f} | Checkpoint 1 passed")

## 2 · FedAvg
Each round: send the global `w` to every site → each trains locally → the server averages the weights **weighted by the number of local samples**. **TODO 2:** `fedavg_round(w, sites, epochs)`.

In [ ]:
def fedavg_round(w, sites, epochs=5, lr=0.5):
    # TODO 2: local weights from every site (local_sgd), then the average weighted by n_i / sum(n)
    return w
def run_fed(sites, rounds=30, epochs=5):
    w = np.zeros(6); curve = []
    for _ in range(rounds): w = fedavg_round(w, sites, epochs); curve.append(auc(yte, sigmoid(add_bias(Xte) @ w)))
    return w, curve

In [ ]:
wf, cf = run_fed(SITES)
Xc = np.vstack([s[0] for s in SITES]); yc = np.concatenate([s[1] for s in SITES])
wc = local_sgd(np.zeros(6), Xc, yc, epochs=150); auc_c = auc(yte, sigmoid(add_bias(Xte) @ wc))
auc_l = [auc(yte, sigmoid(add_bias(Xte) @ local_sgd(np.zeros(6), *s, epochs=150))) for s in SITES]
print(f"federated {cf[-1]:.3f} | centralised (pooled data) {auc_c:.3f} | each hospital alone {np.round(auc_l, 3)}")
assert cf[-1] > 0.75 and cf[-1] > cf[0] - 0.02, "federated model should learn"
plt.plot(cf, label="FedAvg"); plt.axhline(auc_c, ls="--", c="grey", label="centralised"); plt.xlabel("round"); plt.ylabel("test AUC"); plt.legend(); plt.show()
print("Checkpoint 2 passed")

## 3 · Plus: non-IID sites
Now site 3 has a very different population: **prevalence 0.8** instead of 0.3 and a strong feature shift. **TODO 3:** rebuild the third site, rerun `run_fed`, and compare the final AUC and the curve with the IID run. Optional: raise `epochs` per round to 20 and watch the **client drift**.

In [ ]:
# TODO 3: SITES_NIID = same first two sites + a third made with make_site(400, np.array([-1.5, 0.8, 0, 0, 0.]), 0.80, seed=3)
# run run_fed on it (epochs=5 and epochs=20); compare the final AUC with the IID run `cf[-1]`
print("write your code here")

**Reflection.** 1) What exactly leaves each hospital? Could the weights still leak information about patients? (Look up *gradient inversion*; the answer is "yes, in some settings".) 2) Why is the weighted average weighted? 3) When is a federated model *worse* than the best single hospital? 4) Name two non-technical obstacles to a real federated project (governance, data harmonisation, who owns the model).

*Note.* No privacy guarantee here: real systems add secure aggregation and/or differential privacy.